# POS Tagging avec HMM & Viterbi - Version Corrigee et Vectorisee
### Arsene Mekondion

**Objectif** : Etiquetage morphosyntaxique (Parts-of-Speech Tagging) sur le corpus Wall Street Journal en utilisant un Modele de Markov Cache (HMM) et l'algorithme de Viterbi.

**Points cles de cette version :**
- Toutes les fonctions sont **corrigees** (plus de `None`)
- Les matrices de transition/emission sont construites de maniere **vectorisee** avec NumPy
- L'algorithme de Viterbi est **vectorise** : la boucle interne sur les tags precedents est remplacee par des operations matricielles
- Pas de tests unitaires - uniquement le code fonctionnel

In [ ]:
# --- Imports ---
from utils_pos import get_word_tag, preprocess
import pandas as pd
from collections import defaultdict
import numpy as np

In [ ]:
# --- Chargement des donnees ---

# Corpus d'entrainement (WSJ sections 02-21)
with open("./data/WSJ_02-21.pos", 'r') as f:
    training_corpus = f.readlines()

# Vocabulaire
with open("./data/hmm_vocab.txt", 'r') as f:
    voc_l = f.read().split('\n')

# Dictionnaire mot -> index (trie alphabetiquement)
vocab = {word: i for i, word in enumerate(sorted(voc_l))}

# Corpus de test (WSJ section 24) avec tags
with open("./data/WSJ_24.pos", 'r') as f:
    y = f.readlines()

# Corpus de test preprocesse (mots sans tags)
_, prep = preprocess(vocab, "./data/test.words")

print(f"Taille du corpus d'entrainement : {len(training_corpus)}")
print(f"Taille du vocabulaire : {len(vocab)}")
print(f"Taille du corpus de test : {len(prep)}")

## 1 - Construction des dictionnaires de comptage

On parcourt le corpus d'entrainement pour compter :
- **transition_counts** : `(tag_precedent, tag_courant) -> nombre d'occurrences`
- **emission_counts** : `(tag, mot) -> nombre d'occurrences`
- **tag_counts** : `tag -> nombre d'occurrences`

In [ ]:
def create_dictionaries(training_corpus, vocab, verbose=True):
    """
    Parcourt le corpus d'entrainement et construit les dictionnaires
    de comptage pour les transitions, emissions et tags.
    """
    emission_counts = defaultdict(int)
    transition_counts = defaultdict(int)
    tag_counts = defaultdict(int)

    prev_tag = '--s--'

    for i, word_tag in enumerate(training_corpus, 1):
        if i % 50000 == 0 and verbose:
            print(f"word count = {i}")

        # Extraire le mot et le tag de la ligne courante
        word, tag = get_word_tag(word_tag, vocab)

        # Incrementer les compteurs
        transition_counts[(prev_tag, tag)] += 1
        emission_counts[(tag, word)] += 1
        tag_counts[tag] += 1

        prev_tag = tag

    return emission_counts, transition_counts, tag_counts

In [ ]:
emission_counts, transition_counts, tag_counts = create_dictionaries(training_corpus, vocab)

states = sorted(tag_counts.keys())
print(f"Nombre de tags POS (etats) : {len(states)}")
print(f"Tags : {states}\n")

print("Exemples de transitions :")
for ex in list(transition_counts.items())[:3]:
    print(f"  {ex}")

print("\nExemples d'emissions :")
for ex in list(emission_counts.items())[200:203]:
    print(f"  {ex}")

## 1.2 - Prediction naive par emission la plus frequente

Pour chaque mot, on predit le tag POS qui a le plus grand nombre d'emissions.
C'est une baseline simple avant d'utiliser le HMM complet.

**Vectorisation** : on pre-calcule le meilleur tag par mot une seule fois au lieu de boucler sur tous les etats pour chaque mot.

In [ ]:
def predict_pos(prep, y, emission_counts, vocab, states):
    """
    Prediction naive : pour chaque mot, on choisit le tag
    avec le plus grand comptage d'emission.
    
    Optimisation : on pre-calcule un dictionnaire mot -> meilleur tag
    au lieu de boucler sur tous les etats pour chaque mot du corpus.
    """
    # Pre-calcul : pour chaque mot, trouver le tag avec le max d'emissions
    best_tag_for_word = {}
    for (tag, word), count in emission_counts.items():
        if word not in best_tag_for_word or count > best_tag_for_word[word][1]:
            best_tag_for_word[word] = (tag, count)

    num_correct = 0
    total = 0

    for word, y_tup in zip(prep, y):
        y_tup_l = y_tup.split()
        if len(y_tup_l) != 2:
            continue

        true_label = y_tup_l[1]
        total += 1

        if word in vocab and word in best_tag_for_word:
            if best_tag_for_word[word][0] == true_label:
                num_correct += 1

    return num_correct / total

In [ ]:
accuracy_predict_pos = predict_pos(prep, y, emission_counts, vocab, states)
print(f"Precision de la prediction naive (emission) : {accuracy_predict_pos:.4f}")
# Attendu : ~0.8889

## 2 - Modele de Markov Cache (HMM)

### 2.1 - Matrice de transition A (vectorisee)

$$A_{i,j} = \frac{C(tag_i, tag_j) + \alpha}{C(tag_i) + \alpha \times N}$$

**Vectorisation** : au lieu d'une double boucle `O(N^2)` avec des lookups dictionnaire, on remplit la matrice de comptage en une seule passe sur `transition_counts`, puis on applique le lissage en une operation matricielle.

In [ ]:
def create_transition_matrix(alpha, tag_counts, transition_counts):
    """
    Construit la matrice de transition A avec lissage additif.
    Version vectorisee : remplissage par iteration sur les cles existantes,
    puis normalisation matricielle en une seule operation NumPy.
    """
    all_tags = sorted(tag_counts.keys())
    num_tags = len(all_tags)
    tag_to_idx = {tag: i for i, tag in enumerate(all_tags)}

    # Remplir la matrice de comptage (seules les paires observees)
    A = np.zeros((num_tags, num_tags))
    for (prev_tag, tag), count in transition_counts.items():
        A[tag_to_idx[prev_tag], tag_to_idx[tag]] = count

    # Vecteur des comptages par tag (pour chaque ligne)
    tag_count_vec = np.array([tag_counts[tag] for tag in all_tags])

    # Lissage vectorise : (count + alpha) / (count_prev + alpha * N)
    A = (A + alpha) / (tag_count_vec.reshape(-1, 1) + alpha * num_tags)

    return A

In [ ]:
alpha = 0.001
A = create_transition_matrix(alpha, tag_counts, transition_counts)

print(f"A[0,0] : {A[0,0]:.9f}")
print(f"A[3,1] : {A[3,1]:.4f}")
print(f"\nSous-matrice A[30:35, 30:35] :")
print(pd.DataFrame(A[30:35, 30:35], index=states[30:35], columns=states[30:35]))

### 2.2 - Matrice d'emission B (vectorisee)

$$B_{i,j} = \frac{C(tag_i, mot_j) + \alpha}{C(tag_i) + \alpha \times V}$$

Meme strategie de vectorisation que pour A : remplissage sparse puis normalisation matricielle.

In [ ]:
def create_emission_matrix(alpha, tag_counts, emission_counts, vocab):
    """
    Construit la matrice d'emission B avec lissage additif.
    Version vectorisee : remplissage par les cles existantes,
    puis normalisation matricielle.
    """
    all_tags = sorted(tag_counts.keys())
    num_tags = len(all_tags)
    tag_to_idx = {tag: i for i, tag in enumerate(all_tags)}

    # vocab est passe comme liste dans l'appel original
    if isinstance(vocab, dict):
        word_to_idx = vocab
        num_words = len(vocab)
    else:
        word_to_idx = {word: i for i, word in enumerate(vocab)}
        num_words = len(vocab)

    # Remplir la matrice de comptage
    B = np.zeros((num_tags, num_words))
    for (tag, word), count in emission_counts.items():
        if word in word_to_idx:
            B[tag_to_idx[tag], word_to_idx[word]] = count

    # Vecteur des comptages par tag
    tag_count_vec = np.array([tag_counts[tag] for tag in all_tags])

    # Lissage vectorise
    B = (B + alpha) / (tag_count_vec.reshape(-1, 1) + alpha * num_words)

    return B

In [ ]:
B = create_emission_matrix(alpha, tag_counts, emission_counts, list(vocab))

print(f"B[0,0] : {B[0,0]:.9f}")
print(f"B[3,1] : {B[3,1]:.9f}")

# Afficher un extrait pour quelques mots et tags
cidx = ['725', 'adroitly', 'engineers', 'promoted', 'synergy']
cols = [vocab[a] for a in cidx]
rvals = ['CD', 'NN', 'NNS', 'VB', 'RB', 'RP']
rows = [states.index(a) for a in rvals]

print(f"\nExtrait de la matrice B :")
print(pd.DataFrame(B[np.ix_(rows, cols)], index=rvals, columns=cidx))

## 3 - Algorithme de Viterbi (vectorise)

L'algorithme de Viterbi trouve la sequence de tags la plus probable pour une phrase donnee en utilisant la programmation dynamique.

### 3.1 - Initialisation (vectorisee)

$$best\_probs_{i,0} = \log(A_{s,i}) + \log(B_{i, mot_0})$$

Calcul vectorise en une seule ligne NumPy au lieu d'une boucle sur les tags.

In [ ]:
def initialize(states, tag_counts, A, B, corpus, vocab):
    """
    Initialise les matrices best_probs et best_paths pour le premier mot.
    Version vectorisee : calcul de toute la colonne 0 en une operation.
    """
    num_tags = len(tag_counts)
    best_probs = np.zeros((num_tags, len(corpus)))
    best_paths = np.zeros((num_tags, len(corpus)), dtype=int)

    s_idx = states.index("--s--")

    # Index du premier mot dans le vocabulaire
    word_idx = vocab[corpus[0]]

    # Vectorise : log(A[s_idx, :]) + log(B[:, word_idx]) pour tous les tags
    best_probs[:, 0] = np.log(A[s_idx, :]) + np.log(B[:, word_idx])

    return best_probs, best_paths

In [ ]:
best_probs, best_paths = initialize(states, tag_counts, A, B, prep, vocab)
print(f"best_probs[0,0] : {best_probs[0,0]:.4f}")
print(f"best_paths[2,3] : {best_paths[2,3]:.4f}")

### 3.2 - Viterbi Forward (vectorise)

Pour chaque mot `i` et chaque tag courant `j`, on cherche le tag precedent `k` qui maximise :

$$prob(k, j) = best\_probs_{k,i-1} + \log(A_{k,j}) + \log(B_{j, mot_i})$$

**Vectorisation cle** : au lieu de la double boucle `j x k`, on calcule une matrice `(num_tags x num_tags)` en une seule operation broadcasting, puis on extrait le max et argmax par colonne.

```
prob_matrix = best_probs[:, i-1].reshape(-1, 1) + log_A + log_B[:, word_idx].reshape(1, -1)
best_probs[:, i] = max(prob_matrix, axis=0)
best_paths[:, i] = argmax(prob_matrix, axis=0)
```

Cela elimine la boucle interne sur `k` (46 tags) et accelere significativement le calcul.

In [ ]:
def viterbi_forward(A, B, test_corpus, best_probs, best_paths, vocab, verbose=True):
    """
    Passe forward de Viterbi.
    Version vectorisee : les deux boucles internes (j, k) sont remplacees
    par des operations matricielles NumPy avec broadcasting.
    """
    num_tags = best_probs.shape[0]

    # Pre-calculer les log pour eviter de recalculer a chaque iteration
    log_A = np.log(A)
    log_B = np.log(B)

    for i in range(1, len(test_corpus)):
        if i % 5000 == 0 and verbose:
            print("Mots traites : {:>8}".format(i))

        # Index du mot courant dans le vocabulaire
        word_idx = vocab[test_corpus[i]]

        # --- Calcul vectorise ---
        # prob_matrix[k, j] = best_probs[k, i-1] + log(A[k, j]) + log(B[j, word_idx])
        # Shapes : (N,1) + (N,N) + (1,N) => (N,N) via broadcasting
        prob_matrix = (best_probs[:, i - 1].reshape(-1, 1)
                       + log_A
                       + log_B[:, word_idx].reshape(1, -1))

        # Pour chaque tag j (colonne), trouver le meilleur tag precedent k (ligne)
        best_probs[:, i] = np.max(prob_matrix, axis=0)
        best_paths[:, i] = np.argmax(prob_matrix, axis=0)

    return best_probs, best_paths

In [ ]:
# Execution de Viterbi Forward (~30 000 mots)
best_probs, best_paths = viterbi_forward(A, B, prep, best_probs, best_paths, vocab)

print(f"\nbest_probs[0,1] : {best_probs[0,1]:.4f}")
print(f"best_probs[0,4] : {best_probs[0,4]:.4f}")

### 3.3 - Viterbi Backward

On remonte les `best_paths` depuis le dernier mot pour reconstituer la meilleure sequence de tags.

In [ ]:
def viterbi_backward(best_probs, best_paths, corpus, states):
    """
    Passe backward de Viterbi : reconstruit la meilleure sequence de tags
    en remontant les best_paths depuis le dernier mot.
    """
    m = best_paths.shape[1]
    num_tags = best_probs.shape[0]

    # Trouver le meilleur tag pour le dernier mot (vectorise avec argmax)
    z = [None] * m
    pred = [None] * m

    z[m - 1] = int(np.argmax(best_probs[:, m - 1]))
    pred[m - 1] = states[z[m - 1]]

    # Remonter les best_paths
    for i in range(m - 1, 0, -1):
        z[i - 1] = best_paths[z[i], i]
        pred[i - 1] = states[z[i - 1]]

    return pred

In [ ]:
pred = viterbi_backward(best_probs, best_paths, prep, states)
m = len(pred)

print("Predictions pour les derniers mots :")
print(f"  Mots : {prep[-7:m-1]}")
print(f"  Tags : {pred[-7:m-1]}\n")

print("Predictions pour les premiers mots :")
print(f"  Mots : {prep[0:7]}")
print(f"  Tags : {pred[0:7]}")

## 4 - Evaluation de la precision

On compare les predictions de Viterbi avec les labels reels du corpus de test.

**Vectorisation** : on utilise `np.mean` sur des arrays NumPy pour calculer la precision au lieu d'un compteur manuel.

In [ ]:
def compute_accuracy(pred, y):
    """
    Calcule la precision en comparant les predictions aux labels reels.
    Version vectorisee avec arrays NumPy.
    """
    # Extraire les tags valides et les predictions correspondantes
    tags = []
    valid_preds = []

    for prediction, y_line in zip(pred, y):
        parts = y_line.split()
        if len(parts) != 2:
            continue
        tags.append(parts[1])
        valid_preds.append(prediction)

    # Comparaison vectorisee
    return np.mean(np.array(tags) == np.array(valid_preds))

In [ ]:
print(f"Precision de l'algorithme de Viterbi : {compute_accuracy(pred, y):.4f}")
# Attendu : ~0.9531

print(f"\nExemple de prediction :")
print(f"  Mot       : {prep[3]}")
print(f"  Prediction: {pred[3]}")
print(f"  Label reel: {y[3].strip()}")

## Resume - Arsene Mekondion

| Methode | Precision attendue |
|---|---|
| Prediction naive (max emission) | ~0.8889 |
| HMM + Viterbi | ~0.9531 |

**Gains de la vectorisation :**
- `create_transition_matrix` : double boucle `46x46` remplacee par remplissage sparse + normalisation matricielle
- `create_emission_matrix` : double boucle `46 x 23777` remplacee par la meme strategie
- `viterbi_forward` : boucle interne `46x46` par mot remplacee par une multiplication matricielle broadcasting
- `initialize` : boucle sur 46 tags remplacee par une ligne vectorisee
- `compute_accuracy` : comparaison vectorisee avec `np.mean`